# Meteocat Radar Tile Explorer

Interactively play with x, y values and download Meteocat radar images, visualize individual tiles, and stitch them.

In [3]:
# 1. Import Required Libraries
import os
import cv2
import numpy as np
import requests
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import matplotlib.pyplot as plt
import ipywidgets as widgets
display = __import__('IPython.display').display

## 2. Configure Radar Tile Parameters
Set up parameters such as timestamp, zoom, tile size, and allow user input for x and y values.

In [ ]:
# Set up parameters
ZOOM = 7  # Meteocat radar zoom level (7 or 8)
TILE_SIZE = 1024  # px (for radar tiles at zoom 7)

# Default timestamp: 5 minutes ago in Europe/Madrid
RADAR_TIMESTAMP = datetime.now(ZoneInfo("Europe/Madrid")) - timedelta(minutes=5)

# Default x and y ranges for Catalonia (adjust as needed)
default_x_range = [64, 65]
default_y = 80

# Widgets for user input
x_widget = widgets.IntSlider(value=default_x_range[0], min=60, max=70, step=1, description='x')
y_widget = widgets.IntSlider(value=default_y, min=75, max=85, step=1, description='y')
timestamp_widget = widgets.Text(value=RADAR_TIMESTAMP.strftime('%Y-%m-%d %H:%M'), description='Timestamp (local)', placeholder='YYYY-MM-DD HH:MM')
from IPython.display import display as ipy_display
ipy_display(x_widget, y_widget, timestamp_widget)

TypeError: 'module' object is not callable

## 3. Generate Tile URLs
Define a function to generate the Meteocat radar tile URL for given datetime, zoom, x, and y values.

In [ ]:
def get_tile_url(dt, zoom, x, y):
    """Generate Meteocat radar tile URL for given datetime, zoom, x, y."""
    return (
        f"https://static-m.meteo.cat/tiles/radar/"
        f"{dt.year:04d}/{dt.month:02d}/{dt.day:02d}/"
        f"{dt.hour:02d}/{dt.minute:02d}/07/000/000/"
        f"{x:03d}/000/000/{y:03d}.png"
    )

## 4. Download Radar Tiles
Define a function to download a radar tile from a URL and return it as a numpy array.

In [ ]:
def download_tile(url):
    """Download a single tile image from URL and return as numpy array."""
    resp = requests.get(url)
    if resp.status_code == 200:
        arr = np.asarray(bytearray(resp.content), dtype=np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_UNCHANGED)
        return img
    else:
        print(f"Failed to download {url}")
        return None

## 5. Display Downloaded Tiles
Download and display individual radar tiles for selected x and y values using matplotlib.